# 01 — Setup, model load, tool harness probe

**Run on**: 1× A100-80G (or any GPU pod with the network volume mounted at `/workspace`).

**Goal**: confirm the env works end-to-end. After this notebook you should have:
- All packages installed
- Qwen3-4B loaded in bf16 (idle GPU mem reported)
- All three tools (calc, code, search) returning expected outputs on probe inputs
- Tool cache sqlite created at `/workspace/dyna_grpo/tool_cache.sqlite`


In [3]:
# import sys, os
# sys.path.insert(0, str(os.path.abspath(os.path.join(os.getcwd(), '..'))))
# %pip install -q -r ../requirements.txt

In [2]:
import torch
from dyna_grpo.config import MODEL, TOOL_NAMES, PATHS, HUB
print('CUDA available:', torch.cuda.is_available())
print('GPUs:', torch.cuda.device_count())
for k, v in PATHS.items():
    print(f'  {k}: {v}')
print('HF user (for hub push):', HUB.hf_user)

CUDA available: True
GPUs: 1
  models: /workspace/dyna_grpo/models
  data: /workspace/dyna_grpo/data
  traces: /workspace/dyna_grpo/traces
  ckpts: /workspace/dyna_grpo/ckpts
  logs: /workspace/dyna_grpo/logs
  figures: /workspace/dyna_grpo/figures
  tool_cache: /workspace/dyna_grpo/tool_cache.sqlite
HF user (for hub push): genaiquest


## 1. Tool probes

In [4]:
from dyna_grpo.tools import call_tool

for tool, args in [
    ('calc', {'expression': '2**10 + 5*7'}),
    ('code', {'code': 'print(sum(i*i for i in range(11)))'}),
    ('search', {'query': 'NeurIPS 2025 review timeline'}),
]:
    r = call_tool(tool, args)
    print(f'== {tool}: {args} ==')
    print(f'  output: {(r.output or r.error)[:200]}')
    print(f'  latency: {r.latency_ms:.1f} ms')

== calc: {'expression': '2**10 + 5*7'} ==
  output: 1059.0000000000000000
  latency: 0.8 ms
== code: {'code': 'print(sum(i*i for i in range(11)))'} ==
  output: 385

  latency: 235.9 ms
== search: {'query': 'NeurIPS 2025 review timeline'} ==
  output: [{"title": "2025 Dates and Deadlines", "snippet": "NeurIPS 2025 Meeting Dates. The Thirty-Ninth annual conference is held Sun.The NeurIPS Logo above may be used on presentations. Right-click and choos
  latency: 1272.7 ms


In [6]:
# Re-run the same calls — they should be cache hits (latency ~0)
import time
for tool, args in [('calc', {'expression': '2**10 + 5*7'}),
                   ('code', {'code': 'print(sum(i*i for i in range(11)))'})]:
    t0 = time.perf_counter()
    r = call_tool(tool, args)
    print(f'{tool}: {(time.perf_counter()-t0)*1000:.2f} ms (cached)')

calc: 23.49 ms (cached)
code: 24.20 ms (cached)


## 2. Load Qwen3-4B (bf16)

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL.actor_name, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL.actor_name, torch_dtype=torch.bfloat16, device_map='cuda:0',
    trust_remote_code=True)
model.eval()
print('Memory used (GiB):', torch.cuda.memory_allocated() / 1024**3)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Memory used (GiB): 7.492448329925537


In [8]:
import time
from dyna_grpo.tools import system_prompt
prompt = system_prompt(4) + '\n\nUser: What is 17 * 23?\nAssistant: '
enc = tok(prompt, return_tensors='pt').to('cuda:0')
t0 = time.perf_counter()
out = model.generate(**enc, max_new_tokens=120, do_sample=False, pad_token_id=tok.pad_token_id)
print(f'Generation time: {time.perf_counter()-t0:.2f}s')
print(tok.decode(out[0][enc.input_ids.size(1):], skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generation time: 1.21s
17 * 23 = 391
<answer>391</answer>


## 3. End-to-end ReAct rollout sanity


In [9]:
from dyna_grpo.trace_collector import rollout

def gen_fn(ctx, max_new):
    enc = tok(ctx, return_tensors='pt').to('cuda:0')
    o = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                       pad_token_id=tok.pad_token_id)
    return tok.decode(o[0][enc.input_ids.size(1):], skip_special_tokens=True)

traj = rollout('Compute 12345 * 67890 step by step.', gen_fn, max_tool_calls=2)
for i, s in enumerate(traj.segments):
    print(f'[{i}] {s.type}: {(s.text or "")[:120]}')
    if s.obs is not None:
        print(f'     obs ({s.obs_source}): {s.obs[:100]}')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[0] tool_call: 12345 * 67890 can be computed directly using multiplication. Let's calculate it step by step.

First, break down the mul
     obs (real): 838102050.00000000000
[1] gen:  The result of 12345 * 67890 is 838102050. 

<answer>838102050</answer>


✅ If all three sections produced sensible output, you're ready for **Notebook 02**.
